# 13 · Uncertainty and bias adjustment

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/13-uncertainty-and-bias.ipynb)



Browser-only piplite prepares numpy, pandas, matplotlib and Plotly 6.5.0; native kernels use installed scientific packages. Local calculations use bundled helpers and inputs; Plotly rendering can require its versioned CDN and WebGL. Seeded monthly examples are synthetic. The C3S warming-level table gives model windows, Natural Earth provides real generalized coastlines, and the CDS catalogue is metadata. No authenticated CDS request, archive download or native regridding is run.

**Provenance:** `content/13-uncertainty-and-bias.ipynb` at `b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a`; original SHA-256 `8fb00f2a2801dd71ed5fc85c6f09010496816185ce392709ed202ce787b119c7`. Original course material MIT; C3S auxiliary table Apache-2.0, Natural Earth public domain and CDS catalogue metadata CC BY 4.0. Preserve source credits, licenses and artwork provenance.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Original climate-globe illustration introducing Uncertainty and bias adjustment; illustrative, not a data map](assets/atlas-splash.png)

# 13 · Uncertainty and bias adjustment

**Guiding question:** Can a corrected mean still leave the wrong extreme behaviour?

**Data:** Synthetic teaching examples  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-15  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** in-memory tables and figures; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=13-uncertainty-and-bias.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

Yes. A mean shift can align the average while leaving variability and tails
       wrong. Bias adjustment requires a calibration period, an observational reference,
       and a method appropriate to the variable. It cannot eliminate scenario uncertainty
       or create reliable local detail that was absent from the model.

       Sign agreement across models is useful descriptive information, but it is not
       the complete Atlas robustness classification. This lesson deliberately does not
       reproduce official hatching from a single arbitrary agreement threshold.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import importlib
    import piplite
    _atlas_browser_requirements = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "plotly": "plotly==6.5.0"}
    _atlas_browser_package_status = {"retried": [], "verified": []}
    await piplite.install(list(_atlas_browser_requirements.values()))

    def _atlas_missing_browser_packages():
        importlib.invalidate_caches()
        missing = []
        for module in _atlas_browser_requirements:
            try:
                importlib.import_module(module)
            except ModuleNotFoundError as error:
                if error.name != module:
                    raise RuntimeError(f"Browser package {module} has a missing dependency ({error.name}); preparation stopped without retrying an unrelated failure.") from error
                missing.append(module)
        return missing

    _atlas_missing = _atlas_missing_browser_packages()
    if _atlas_missing:
        _atlas_browser_package_status["retried"] = list(_atlas_missing)
        print("ATLAS_PACKAGE_RETRY", _atlas_missing, flush=True)
        # One bounded retrieval retry, only for directly confirmed missing modules.
        await piplite.install([_atlas_browser_requirements[module] for module in _atlas_missing])
        _atlas_missing = _atlas_missing_browser_packages()
    if _atlas_missing:
        raise RuntimeError("Browser package preparation failed after one missing-module retry: " + ", ".join(_atlas_missing) + ". Check package/network diagnostics, then restart the kernel.")
    _atlas_browser_package_status["verified"] = list(_atlas_browser_requirements)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Separate mean bias from tail error

In [2]:
rng = np.random.default_rng(13)
reference = rng.normal(30, 3, 5000)
model = rng.normal(32, 5, 5000)
adjusted = model - model.mean() + reference.mean()
summary = pd.DataFrame({"series": ["reference", "model", "mean-shifted model"],
    "mean": [x.mean() for x in [reference, model, adjusted]],
    "p95": [np.quantile(x, .95) for x in [reference, model, adjusted]],
    "fraction above 35 °C": [(x > 35).mean() for x in [reference, model, adjusted]]})
display(summary.round(3))
fig = go.Figure()
for label, values in [("Reference", reference), ("Model", model), ("Mean shifted", adjusted)]:
    fig.add_histogram(x=values, histnorm="probability density", opacity=.45, name=label)
fig.update_layout(barmode="overlay", title="Mean correction leaves tail differences · SYNTHETIC", xaxis_title="Daily maximum temperature (°C)", yaxis_title="Density (1/°C)")
show(fig)

,series,mean,p95,fraction above 35 °C
0,reference,30.078,34.925,0.047
1,model,31.964,40.041,0.271
2,mean-shifted model,30.078,38.154,0.159


Figure({
    'data': [{'histnorm': 'probability density',
              'name': 'Reference',
              'opacity': 0.45,
              'type': 'histogram',
              'x': {'bdata': ('6ScUenm9QUASZt5R18M0QBAkYoXlb0' ... 'wDUD9AN7vsbwCtO0BSqZb7Tx1BQA=='),
                    'dtype': 'f8'}},
             {'histnorm': 'probability density',
              'name': 'Model',
              'opacity': 0.45,
              'type': 'histogram',
              'x': {'bdata': ('7zNGgBrbOkBNHFZBz4g+QPqNAO6Qgz' ... 'BqukJA9XkB1Y01P0DP7rn0r0FCQA=='),
                    'dtype': 'f8'}},
             {'histnorm': 'probability density',
              'name': 'Mean shifted',
              'opacity': 0.45,
              'type': 'histogram',
              'x': {'bdata': ('JNf0Zzf4OECCvwQp7KU8QC8xr9WtoD' ... 'T5yEFAKh2wvKpSPUBqQJFoPlBBQA=='),
                    'dtype': 'f8'}}],
    'layout': {'barmode': 'overlay',
               'font': {'size': 14},
               'height': 470,
               'margin': {'b': 65, 'l': 65, 'r': 45, 't': 90},
               'template': '...',
               'title': {'text': 'Mean correction leaves tail differences · SYNTHETIC'},
               'xaxis': {'title': {'text': 'Daily maximum temperature (°C)'}},
               'yaxis': {'title': {'text': 'Density (1/°C)'}}}
})

## Count sign agreement honestly

In [3]:
changes = np.array([-0.3, 0.2, 0.5, 0.7, 0.8, 1.0, 1.2, 1.5])
print("Invented members with positive change:", (changes > 0).sum(), "of", len(changes))
print("This fraction is not a probability, a p-value, or the official robustness mask.")

Invented members with positive change: 7 of 8
This fraction is not a probability, a p-value, or the official robustness mask.


## Try it and explain

Why is evaluating the adjustment on the same reference sample not independent validation?

**Interpretation check:** The adjustment was tuned using that sample. Validate on held-out periods and examine tails, seasonality, and physical consistency, while acknowledging climate non-stationarity.

## Transfer to the official Atlas

Read the exact Atlas uncertainty legend for a trend and for a projection change. Explain why they use different evidence.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)
- [Bias adjustment user tools](https://github.com/ecmwf-projects/c3s-atlas/blob/main/book/notebooks/tx35bals.ipynb)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/credits.md) and [data guide](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/data-guide.md).

**Continue:** [Regions, masks, and area weighting](14-regions-and-weights.ipynb)